# 04 — Phase 4: Feature Engineering (merged)

**Purpose.** Runs the merged `sql/005_phase4_features.sql`, which builds `quarterly_modeling_table` (one row per county-quarter: the decline label plus six features measured one quarter earlier), and then `sql/003b_cv_fold_membership.sql`, which rebuilds the `cv_fold_rows` view that reads it. The notebook checks the table, compares it with Jolenn's pandas feature matrix, and shows how the two test-set conventions differ.

**The six features** (Jolenn's column names, used by her model outputs)

| Column | Meaning |
|---|---|
| `momentum_1m`, `momentum_3m`, `momentum_6m` | Relative change in the county's ZHVI over 1, 3 and 6 months, measured at the end of the quarter before the label quarter |
| `mortgage_rate_level` | National 30-year rate at that quarter end |
| `mortgage_rate_roc` | 12-month change in that rate, in **percentage points** |
| `rate_roc_x_momentum` | `mortgage_rate_roc` x `momentum_3m` |

**Decisions carried into this merged phase**

| Decision | Choice | Why |
|---|---|---|
| Rate-of-change | Percentage points, `rate(t) - rate(t-12)` | Same as Phase 3 and Jolenn's features |
| Interaction term | Rate change x **3-month** momentum | Jolenn's choice; 3 months is one quarter, the prediction horizon. The earlier SQL used 1-month |
| Alignment | Features from the preceding quarter | Predict one quarter ahead |
| Missing values | Drop a row if ANY of the six features is missing | The earlier SQL only checked two of them |
| Row dating | By **label** quarter (the convention behind the Phase 0 counts) | Jolenn's matrix is dated by feature quarter; section 7 shows the effect |
| Label rule | Both quarter-end months must have a ZHVI value (quarter-end month only, for the label and the features) | Same rule as Jolenn's `labels.py` and `sql/004`. The earlier SQL used the last month with a value, which gave 448 Train rows whose label spanned two, or four to five, months (about 25% declines against 4.9% overall) |

**Prerequisites.** Phase 2's `003_phase2_split.sql` has run (it builds `monthly_split` and `cv_folds`). Phase 3 is not required.

**Expected interaction sign (stated in advance, checked in Phase 5): negative.** The logistic-regression coefficient on `rate_roc_x_momentum` should be negative, meaning rate rises hurt more when momentum is weak. With about 82 independent quarters behind the rate features (see notebook 03), Phase 5 reads this sign as exploratory.

## 1. Confirm the kernel

In [1]:
import sys
import numpy as np
import pandas as pd
print(sys.executable)

<project folder>\.venv\Scripts\python.exe


## 2. Connect to Postgres

Same convention as notebooks 02 and 03.

In [2]:
import os
from pathlib import Path

from sqlalchemy import create_engine, text

try:
    from housing_risk.config import get_engine
    engine = get_engine()
except ImportError:
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv(usecwd=True), override=True)
    engine = create_engine(
        "postgresql+psycopg2://{u}:{p}@{h}:{port}/{d}".format(
            u=os.environ["PGUSER"], p=os.environ["PGPASSWORD"],
            h=os.environ.get("PGHOST", "localhost"),
            port=os.environ.get("PGPORT", "5432"),
            d=os.environ.get("PGDATABASE", "housing_risk_team"),
        )
    )

with engine.connect() as conn:
    print(conn.execute(text("SELECT current_database(), version();")).fetchone())
    for name in ("monthly_split", "cv_folds"):
        assert conn.execute(text("SELECT to_regclass(:n)"), {"n": name}).scalar(), (
            f"STOP: {name} not found -- run sql/003_phase2_split.sql first."
        )
print("monthly_split and cv_folds found.")

('housing_risk_team', 'PostgreSQL 18.6 on x86_64-windows, compiled by msvc-19.44.35228, 64-bit')
monthly_split and cv_folds found.


## 3. Run `005_phase4_features.sql`, then `003b_cv_fold_membership.sql`

`005` rebuilds `quarterly_modeling_table` and drops the `cv_fold_rows` view with it, so `003b` is run again straight afterwards. Both files can be re-run safely.

In [3]:
def find_sql(name):
    for base in (Path("sql"), Path("..") / "sql"):
        if (base / name).exists():
            return base / name
    raise FileNotFoundError(f"{name} not found in ./sql or ../sql")


def run_sql_file(name):
    path = find_sql(name)
    raw = engine.raw_connection()
    try:
        cur = raw.cursor()
        cur.execute(open(path).read())
        raw.commit()
    except Exception:
        raw.rollback()
        raise
    finally:
        raw.close()
    print(f"ran {path}")

run_sql_file("005_phase4_features.sql")
run_sql_file("003b_cv_fold_membership.sql")

ran ..\sql\005_phase4_features.sql
ran ..\sql\003b_cv_fold_membership.sql


## 3b. Re-check the fold view

`003b` has just rebuilt `cv_fold_rows` on the new table, so the checks from notebook 02 are repeated: every validation fold keeps at least 20 positives, and no Embargo or Test row can enter a fold.

In [4]:
fold_summary = pd.read_sql(
    """
    SELECT fold_id, fold_role, COUNT(*) AS n_rows, SUM(decline_label) AS n_positives
    FROM cv_fold_rows GROUP BY fold_id, fold_role ORDER BY fold_id, fold_role
    """, engine)
MIN_FOLD_POSITIVES = 20
val = fold_summary[fold_summary["fold_role"] == "val"]
assert val["n_positives"].min() >= MIN_FOLD_POSITIVES, (
    f"STOP: a validation fold has fewer than {MIN_FOLD_POSITIVES} positives")
leak = pd.read_sql(
    """
    SELECT COUNT(*) FROM cv_fold_rows r
    JOIN quarterly_modeling_table m USING (county_fips, quarter)
    WHERE m.split <> 'train'
    """, engine).iloc[0, 0]
assert leak == 0, f"STOP: {leak} fold rows come from Embargo/Test"
print(f"{len(val)} validation folds; the lowest has {int(val['n_positives'].min())} positives "
      f"(minimum {MIN_FOLD_POSITIVES}); no Embargo/Test rows appear in any fold.")

11 validation folds; the lowest has 71 positives (minimum 20); no Embargo/Test rows appear in any fold.


## 4. Row counts per split

Compared with the two reference points: the Phase 0 provenance figures (they count label-quarter rows before the feature history is required, so ours are lower) and Jolenn's Train feature matrix (164,460 rows, 3,009 counties, 4.85% positive).

In [5]:
counts = pd.read_sql(
    """
    SELECT split, COUNT(1) AS n_rows, COUNT(DISTINCT county_fips) AS n_counties,
           ROUND(100.0 * AVG(decline_label), 2) AS positive_rate_pct
    FROM quarterly_modeling_table GROUP BY split ORDER BY split
    """, engine)
print(counts.to_string(index=False))
print()
train = counts.set_index("split").loc["train"]
print(f"SQL Train:                     {int(train.n_rows):,} rows, {int(train.n_counties):,} counties, {train.positive_rate_pct}% positive")
print("Jolenn's Train feature matrix: 164,460 rows, 3,009 counties, 4.85% positive")
print("Phase 0 documented (labels):   Train 173,824 / Embargo 3,021 / Test 52,087")
print("Train matches Jolenn's matrix:     ", int(train.n_rows) == 164_460 and int(train.n_counties) == 3_009)

     split  n_rows  n_counties  positive_rate_pct
   embargo    3020        3020               0.96
      test   51980        3071               8.11
     train  164460        3009               4.85
unassigned    3070        3070               4.27

SQL Train:                     164,460 rows, 3,009 counties, 4.85% positive
Jolenn's Train feature matrix: 164,460 rows, 3,009 counties, 4.85% positive
Phase 0 documented (labels):   Train 173,824 / Embargo 3,021 / Test 52,087
Train matches Jolenn's matrix:      True


## 5. Table checks

No missing features, one row per county and quarter, features measured at a quarter end, and the interaction equal to the plain product.

In [6]:
feat_cols = ["momentum_1m", "momentum_3m", "momentum_6m", "mortgage_rate_level", "mortgage_rate_roc", "rate_roc_x_momentum"]
mt = pd.read_sql("SELECT county_fips, quarter, feature_quarter_end, split, decline_label, " + ", ".join(feat_cols) +
                 " FROM quarterly_modeling_table", engine)
mt["quarter"] = pd.to_datetime(mt["quarter"]); mt["feature_quarter_end"] = pd.to_datetime(mt["feature_quarter_end"])

assert mt[feat_cols].isna().sum().sum() == 0, "STOP: missing feature values in quarterly_modeling_table"
assert not mt.duplicated(["county_fips", "quarter"]).any(), "STOP: more than one row per county and quarter"
assert mt["feature_quarter_end"].dt.month.isin([3, 6, 9, 12]).all(), "STOP: features not measured at a quarter end"
assert ((mt["quarter"] - mt["feature_quarter_end"]).dt.days == 1).all(), "STOP: features are not from the preceding quarter"
prod = mt["mortgage_rate_roc"] * mt["momentum_3m"]
assert (mt["rate_roc_x_momentum"] - prod).abs().max() < 1e-9, "STOP: interaction is not mortgage_rate_roc x momentum_3m"
print(f"{len(mt):,} rows: no missing features, one row per county and quarter, features from the preceding quarter end, "
      "interaction = rate change x 3-month momentum.")

222,530 rows: no missing features, one row per county and quarter, features from the preceding quarter end, interaction = rate change x 3-month momentum.


## 6. Feature summary on the Train split

Saved to `../outputs/phase4_feature_summary.csv`.

In [7]:
summary = mt.loc[mt["split"] == "train", feat_cols].describe().T[["mean", "std", "min", "50%", "max"]]
summary.loc["decline_label (share)"] = [mt.loc[mt["split"] == "train", "decline_label"].mean(), np.nan, 0, np.nan, 1]
os.makedirs("../outputs", exist_ok=True)
summary.to_csv("../outputs/phase4_feature_summary.csv")
summary.round(4)

,mean,std,min,50%,max
momentum_1m,0.0034,0.0071,-0.0810,0.0031,0.0730
momentum_3m,0.0100,0.0197,-0.2018,0.0096,0.2284
momentum_6m,0.0197,0.0360,-0.3058,0.0191,0.3738
mortgage_rate_level,4.4938,1.0825,2.6840,4.1960,7.1600
mortgage_rate_roc,-0.2171,0.5813,-1.4200,-0.2480,1.1125
rate_roc_x_momentum,-0.0019,0.0139,-0.2200,-0.0004,0.1776
decline_label (share),0.0485,NaN,0.0000,NaN,1.0000


## 7. Test-set dating convention

Our table is dated by **label** quarter: the Test split holds labels from 2022-Q1 to 2026-Q1 (17 quarters). Jolenn's matrix is dated by **feature** quarter: a row belongs to Test if its features fall inside the test window, and its last feature quarter has no label inside the window. That leaves 16 label quarters, starting at 2022-Q2, because the 2022-Q1 label takes its features from the embargo quarter (2021-Q4). Applying her convention to our table gives her 48,960 rows.

In [8]:
test = mt[mt["split"] == "test"]
n_label_dating = len(test)
n_feature_dating = int(((test["feature_quarter_end"] >= "2022-01-01") & (test["feature_quarter_end"] <= "2026-03-31")).sum())
print(f"Test rows, dated by label quarter (our convention):   {n_label_dating:,}  ({test['quarter'].nunique()} label quarters)")
print(f"Same rows, dated by feature quarter (Jolenn's):       {n_feature_dating:,}  "
      f"({test.loc[test['feature_quarter_end'] >= '2022-01-01', 'quarter'].nunique()} label quarters)")
print("Jolenn's Test feature matrix:                          48,960 rows")
print(f"Our table under her convention minus her matrix:       {n_feature_dating - 48960:+,} rows (near zero means the dating convention explains the gap)")

Test rows, dated by label quarter (our convention):   51,980  (17 label quarters)
Same rows, dated by feature quarter (Jolenn's):       48,960  (16 label quarters)
Jolenn's Test feature matrix:                          48,960 rows
Our table under her convention minus her matrix:       +0 rows (near zero means the dating convention explains the gap)


## 8. Row-by-row comparison with Jolenn's feature matrix (optional)

Needs her `train_features.csv` (from her `data/processed/`, which git ignores). Ask her to send it, and save it as `data/processed/jolenn_train_features.csv` on this machine. **This comparison needs her pandas matrix saved locally, so it cannot be run from the repo alone.** The cell is skipped when the file is missing. The two tables are joined on county and feature quarter end. With the quarter-end label rule both builds hold exactly the same rows, so the cell stops with an error unless there are no rows on only one side, the feature values agree (below 1e-9) and no label differs.

In [9]:
JOLENN_CSV = Path("../data/processed/jolenn_train_features.csv")
if not JOLENN_CSV.exists():
    print(f"{JOLENN_CSV} not found. Ask Jolenn for her data/processed/train_features.csv and save it there to run this comparison.")
else:
    jl = pd.read_csv(JOLENN_CSV, dtype={"county_fips": str}, parse_dates=["month"])
    ours = mt[mt["split"] == "train"].rename(columns={"feature_quarter_end": "month"})
    m = ours.merge(jl, on=["county_fips", "month"], how="outer", suffixes=("_sql", "_pd"), indicator=True)
    both = m[m["_merge"] == "both"]
    only_sql, only_pd = (m["_merge"] == "left_only").sum(), (m["_merge"] == "right_only").sum()
    print(f"Rows in both: {len(both):,} | only in our SQL table: {only_sql:,} | only in Jolenn's matrix: {only_pd:,}")
    print(f"Our extra rows as a share of our Train rows: {100 * only_sql / len(ours):.2f}%")
    diffs = {f: float((both[f + '_sql'] - both[f + '_pd']).abs().max()) for f in feat_cols}
    print("Largest absolute difference on matched rows, by feature:")
    for f, d in diffs.items():
        print(f"  {f:22s} {d:.2e}")
    print(f"Label mismatches on matched rows: {int((both['decline_label_sql'] != both['decline_label_pd']).sum()):,}")
    assert only_sql == 0 and only_pd == 0, "STOP: the SQL table and Jolenn's matrix do not hold the same rows"
    assert max(diffs.values()) < 1e-9, "STOP: feature values differ from Jolenn's matrix"
    assert int((both["decline_label_sql"] != both["decline_label_pd"]).sum()) == 0, "STOP: labels differ from Jolenn's matrix"
    print("The two builds match exactly: same rows, same feature values, same labels.")

Rows in both: 164,460 | only in our SQL table: 0 | only in Jolenn's matrix: 0
Our extra rows as a share of our Train rows: 0.00%
Largest absolute difference on matched rows, by feature:
  momentum_1m            7.65e-16
  momentum_3m            7.60e-16
  momentum_6m            7.58e-16
  mortgage_rate_level    8.88e-16
  mortgage_rate_roc      2.22e-16
  rate_roc_x_momentum    7.22e-16
Label mismatches on matched rows: 0
The two builds match exactly: same rows, same feature values, same labels.


In [10]:
# Rows that exist on only one side (expected: none)
if JOLENN_CSV.exists():
    one_side = m[m["_merge"] != "both"]
    if len(one_side) == 0:
        print("No rows exist on only one side.")
    else:
        print(one_side["_merge"].value_counts().to_string())

No rows exist on only one side.


### 8b. Summary check against Jolenn's `train_features_meta.json`

A lighter check that does not need her full matrix. Her notebook 07 writes `outputs/train_features_meta.json` with the size of her Train matrix. Save a copy as `data/processed/jolenn_train_features_meta.json` on this machine. The cell is skipped when the file is missing, and stops with an error if the Train rows, counties or positive rate differ.

In [11]:
import json

JOLENN_META = Path("../data/processed/jolenn_train_features_meta.json")
if not JOLENN_META.exists():
    print(f"{JOLENN_META} not found. Save a copy of Jolenn's outputs/train_features_meta.json there to run this check.")
else:
    meta = json.loads(JOLENN_META.read_text())
    t = mt[mt["split"] == "train"]
    n_rows, n_cty, rate = len(t), t["county_fips"].nunique(), 100 * t["decline_label"].mean()
    print(f"Rows:      ours {n_rows:,} | Jolenn {meta['n_rows']:,}")
    print(f"Counties:  ours {n_cty:,} | Jolenn {meta['n_counties']:,}")
    print(f"Positives: ours {rate:.6f}% | Jolenn {meta['positive_rate_pct']:.6f}%")
    assert n_rows == meta["n_rows"] and n_cty == meta["n_counties"] and abs(rate - meta["positive_rate_pct"]) < 1e-9, \
        "STOP: our Train summary differs from Jolenn's train_features_meta.json"
    print("Train rows, counties and positive rate match Jolenn's summary exactly.")

Rows:      ours 164,460 | Jolenn 164,460
Counties:  ours 3,009 | Jolenn 3,009
Positives: ours 4.851636% | Jolenn 4.851636%
Train rows, counties and positive rate match Jolenn's summary exactly.


## 9. Findings

- **The merged table is clean.** 222,530 rows across the four splits (Train 164,460, Embargo 3,020, Test 51,980, Unassigned 3,070), no missing features, one row per county and quarter, and features measured at the preceding quarter end. The interaction is the plain product of the rate change and the 3-month momentum.
- **Train is identical in size to Jolenn's matrix.** 164,460 rows, 3,009 counties and 4.85% positive, the same as her 164,460 rows, 3,009 counties and 4.85%. Both cover the same counties, so there is no county filter on either side.
- **Row by row, the two builds are identical.** Every one of Jolenn's 164,460 rows is in our table and our table has no others. The largest feature difference is 8.9e-16 (floating-point noise) and there are 0 label mismatches. Her `train_features_meta.json` also matches exactly: 164,460 rows, 3,009 counties and a positive rate of 4.851636%.
- **The test-set gap is fully explained by the dating convention.** Our Test split has 51,980 rows (17 label quarters). Dated by feature quarter, as Jolenn's matrix is, the same rows number 48,960 (16 label quarters), exactly her 48,960.
- **The fold view is intact after the rebuild.** All 11 validation folds keep at least 71 positives (the minimum required is 20), and no Embargo or Test row enters any fold (Section 3b).
- **The label rule matters.** Train went from 165,269 rows (earlier SQL, which checked only two of the six features) to 164,908 (any missing feature drops the row) to 164,460 (both quarter-end months required). The last step removed 448 rows whose label spanned two, or four to five, months and held about 25% declines. No model input holds an empty value, and every label is a true quarter-to-quarter change. The unfinished 2026-Q3 no longer produces rows, so Unassigned is one quarter (3,070 rows).
- **Open for Phase 5:** the expected negative sign of the interaction term, which is read as exploratory because the rate features carry only about 82 independent quarters of information, and whether 1-month or 6-month momentum interacts better than 3-month.